In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer 
import re


def generate_submission_file(folder_path, models_dict, feature_sets, output_file='submission.csv'):
    search_path = os.path.join(folder_path, "*.csv")
    all_files = glob.glob(search_path)
    
    # --- 1. ORDINAMENTO NUMERICO DEI FILE ---
    # Serve per ordinare: test_1, test_2 ... test_10, test_11
    def extract_number(filepath):
        filename = os.path.basename(filepath)
        match = re.search(r'\d+', filename)
        return int(match.group()) if match else 0

    all_files_sorted = sorted(all_files, key=extract_number)

    results = []
    print(f"\n--- GENERAZIONE SUBMISSION FILE ({len(all_files_sorted)} files) ---")
    
    for file_path in all_files_sorted:
        filename = os.path.basename(file_path)
        
        # --- 2. NOME FILE PULITO ---
        # Togliamo il ".csv" per la colonna 'file' (es. "test_0")
        file_id = filename.replace('.csv', '') 
        
        try: df_test_raw = pd.read_csv(file_path)
        except: continue
        if df_test_raw.empty: continue
        
        # Preprocessing (Fisica al volo)
        df_test_raw = harmonize_columns(df_test_raw)
        df_mech = prepare_mechanical_data(df_test_raw)
        df_wash = prepare_wash_data(df_test_raw)
        
        data_map = {'HPT': df_mech, 'HPC': df_mech, 'WW': df_wash}
        row_pred = {'file': file_id} # Prima colonna con nome corretto
        
        for ctype in ['HPT', 'HPC', 'WW']:
            model = models_dict[ctype]['model']
            margin = models_dict[ctype]['margin']
            df = data_map[ctype]
            features = feature_sets[ctype]
            
            for f in features: 
                if f not in df.columns: df[f] = 0
            
            X_test = df[features]
            
            # Protezione Infinito (fondamentale per evitare crash e numeri giganti)
            X_test = X_test.replace([np.inf, -np.inf], np.nan)
            
            imputer = SimpleImputer(strategy='mean')
            try:
                X_clean = pd.DataFrame(imputer.fit_transform(X_test), columns=features)
                X_clean = X_clean.fillna(0) # Ultima sicurezza
                
                pred_raw = model.predict(X_clean)
                pred_safe = np.maximum(pred_raw - margin, 0)
                final_val = pred_safe[-1] if len(pred_safe) > 0 else 0
                row_pred[ctype] = final_val
            except Exception as e: 
                print(f"Errore su {filename} ({ctype}): {e}")
                row_pred[ctype] = 0
                
        results.append(row_pred)
        
    # --- 3. FORMATTAZIONE ESATTA PER LA COMPETIZIONE ---
    df_submission = pd.DataFrame(results)
    
    # Rinominiamo le colonne interne con i nomi ufficiali richiesti
    df_submission = df_submission.rename(columns={
        'WW': 'Cycles_to_WW',
        'HPC': 'Cycles_to_HPC_SV',
        'HPT': 'Cycles_to_HPT_SV'
    })
    
    # Arrotondiamo tutto a numero intero senza virgole
    for col in ['Cycles_to_WW', 'Cycles_to_HPC_SV', 'Cycles_to_HPT_SV']:
        df_submission[col] = df_submission[col].round(0).astype(int)
        
    # Imponiamo l'ORDINE ESATTO delle colonne del template
    df_submission = df_submission[['file', 'Cycles_to_WW', 'Cycles_to_HPC_SV', 'Cycles_to_HPT_SV']]
    
    # Salvataggio in formato standard
    df_submission.to_csv(output_file, index=False)
    print(f"File salvato e pronto per l'upload: {output_file}")
    
    # Stampiamo le prime 5 righe per verificare che sia perfetto!
    print(df_submission.head())


# ==============================================================================
# 0. MOTORE FISICO (Add Physics Features)
# ==============================================================================
def harmonize_columns(df):
    """Normalizza i nomi delle colonne"""
    rename_map = {
        'Cycles': 'Cycles_Since_New', 'Cycle': 'Cycles_Since_New',
        'Altitude': 'Sensed_Altitude', 'Mach': 'Sensed_Mach',
        'TRA': 'Sensed_TRA', 'T2': 'Sensed_T2', 'T24': 'Sensed_T24',
        'T30': 'Sensed_T3', 'T48': 'Sensed_T45', 'T50': 'Sensed_T5',
        'P15': 'Sensed_P15', 'P2': 'Sensed_P2', 'P21': 'Sensed_P21',
        'P24': 'Sensed_P24', 'Ps30': 'Sensed_Ps3', 'P40': 'Sensed_P40',
        'P50': 'Sensed_P50', 'HPC_SV': 'Cycles_to_HPC_SV',
        'HPT_SV': 'Cycles_to_HPT_SV', 'WW': 'Cycles_to_WW'
    }
    df = df.rename(columns=rename_map)
    # Fallback per prefissi mancanti
    if 'Sensed_Altitude' not in df.columns and 'Altitude' not in df.columns:
         for col in df.columns:
             if col not in ['ESN', 'Cycles_Since_New', 'Snapshot', 'File_ID']:
                 if not col.startswith('Sensed_') and not col.startswith('Phy_'):
                     df.rename(columns={col: f'Sensed_{col}'}, inplace=True)
    return df

def add_physics_features(df):
    """
    Applica le leggi della termodinamica al volo.
    Fondamentale per la Validazione che non ha queste colonne pre-calcolate.
    """
    df = df.copy()
    STD_TEMP = 288.15; STD_PRES = 14.696; GAMMA_AIR = 1.4

    # Check colonne minime necessarie
    if 'Sensed_T25' in df.columns and 'Sensed_Pt2' in df.columns:
        theta = (df['Sensed_T25'] + 273.15) / STD_TEMP 
        delta = df['Sensed_Pt2'] / STD_PRES
        theta = theta.replace(0, 1); delta = delta.replace(0, 1)

        # 1. Parametri Corretti
        if 'Sensed_Core_Speed' in df.columns:
            df['Phy_Core_Speed_Corr'] = df['Sensed_Core_Speed'] / np.sqrt(theta)
        if 'Sensed_WFuel' in df.columns:
            df['Phy_WFuel_Corr'] = df['Sensed_WFuel'] / (delta * np.sqrt(theta))
        if 'Sensed_T45' in df.columns:
            df['Phy_T45_Corr'] = (df['Sensed_T45'] + 273.15) / theta

        # 2. Efficienza Isentropica (Compressore)
        if 'Sensed_T3' in df.columns and 'Sensed_Ps3' in df.columns:
            T_in_K = df['Sensed_T25'] + 273.15
            T_out_K = df['Sensed_T3'] + 273.15
            P_in = df['Sensed_Pt2']
            P_out = df['Sensed_Ps3']
            
            pr = P_out / P_in
            k = (GAMMA_AIR - 1) / GAMMA_AIR
            T_iso = T_in_K * (pr ** k)
            df['Phy_Compressor_Eff'] = (T_iso - T_in_K) / (T_out_K - T_in_K)

    # 3. Heat Index
    if 'Sensed_T45' in df.columns and 'Sensed_Ps3' in df.columns:
        df['Phy_Heat_Index'] = df['Sensed_T45'] / df['Sensed_Ps3']

    return df

# ==============================================================================
# 1. PREPROCESSING (PIPELINE AGGIORNATA)
# ==============================================================================
def prepare_mechanical_data(df):
    df = df.copy()
    df = harmonize_columns(df)
    
    # 1. Filtro Crociera (Solo se possibile)
    if 'Sensed_Altitude' in df.columns:
        df = df[df['Sensed_Altitude'] > 20000].copy()
    
    # 2. APPLICAZIONE FISICA (Cruciale per Validation Files)
    df = add_physics_features(df)

    # 3. Selezione Feature (Usiamo quelle fisiche ORA!)
    # Se il file ha i residui pre-calcolati (Train), usiamo quelli se vogliamo, 
    # ma per coerenza con la validazione (che non ha residui storici), 
    # usiamo le Feature Fisiche Corrette (Phy_...) che sono potenti uguali.
    phy_cols = ['Phy_T45_Corr', 'Phy_Compressor_Eff', 'Phy_Heat_Index', 'Phy_Core_Speed_Corr']
    raw_cols = ['Sensed_Ps3', 'Sensed_T3'] # Backup
    
    use_cols = [c for c in phy_cols + raw_cols if c in df.columns]
    
    agg_dict = {col: 'mean' for col in use_cols}
    targets = ['Cycles_to_HPC_SV', 'Cycles_to_HPT_SV']
    for t in targets:
        if t in df.columns: agg_dict[t] = 'first'
        
    if 'ESN' in df.columns:
        df_grouped = df.groupby(['ESN', 'Cycles_Since_New']).agg(agg_dict).reset_index()
        df_grouped = df_grouped.sort_values(['ESN', 'Cycles_Since_New'])
        
        for col in use_cols:
            df_grouped[f"{col}_smooth"] = df_grouped.groupby('ESN')[col].transform(
                lambda x: x.rolling(window=10, min_periods=1).mean()
            )
        df_grouped = df_grouped.ffill().bfill().fillna(0)
        return df_grouped
    return pd.DataFrame()

def prepare_wash_data(df):
    df = df.copy()
    df = harmonize_columns(df)
    
    # Filtro alta potenza (Solo se c'è il sensore)
    if 'Sensed_Core_Speed' in df.columns:
        df = df[df['Sensed_Core_Speed'] > 8000].copy()

    # 1. FISICA (Manteniamo le feature fisiche che sono ottime)
    df = add_physics_features(df)
    
    # 2. Selezione Feature
    # Aggiungiamo 'Sensed_WFuel' grezzo perché a volte la fisica "corregge troppo" il segnale sporco
    phy_cols = ['Phy_Compressor_Eff', 'Phy_Heat_Index', 'Phy_WFuel_Corr']
    raw_backup = ['Sensed_WFuel', 'Sensed_T45'] 
    use_cols = [c for c in phy_cols + raw_backup if c in df.columns]
    
    agg_dict = {}
    for c in use_cols: agg_dict[c] = ['mean', 'max']
    
    # Queste colonne servono per il calcolo del reset clock
    if 'Cycles_to_WW' in df.columns: agg_dict['Cycles_to_WW'] = 'first'
    if 'Cumulative_WWs' in df.columns: agg_dict['Cumulative_WWs'] = 'max' # <--- IMPORTANTE
    
    if 'ESN' in df.columns:
        df_grouped = df.groupby(['ESN', 'Cycles_Since_New']).agg(agg_dict)
        
        new_cols = []
        feature_cols = []
        for c, s in df_grouped.columns:
            if c in ['Cycles_to_WW', 'Cumulative_WWs'] or s == '': new_cols.append(c)
            else: 
                name = f"{c}_{s}"
                new_cols.append(name)
                feature_cols.append(name)
        
        df_grouped.columns = new_cols
        df_grouped = df_grouped.reset_index()
        df_grouped = df_grouped.sort_values(['ESN', 'Cycles_Since_New'])

        # --- IL CUORE DEL WW: RESET CLOCK (RIPRISTINATO) ---
        if 'Cumulative_WWs' in df_grouped.columns:
             # 1. Identifica dove cambia il numero di lavaggi
             df_grouped['WW_Change'] = df_grouped.groupby('ESN')['Cumulative_WWs'].diff().fillna(0)
             # 2. Crea un ID univoco per ogni intervallo tra due lavaggi
             df_grouped['Wash_Session_ID'] = df_grouped.groupby('ESN')['WW_Change'].cumsum()
             # 3. Conta da 0 a partire dall'ultimo lavaggio
             df_grouped['Cycles_Since_Last_Wash'] = df_grouped.groupby(['ESN', 'Wash_Session_ID']).cumcount()
             
             # Aggiungiamo questa feature potentissima alla lista
             feature_cols.append('Cycles_Since_Last_Wash')

        # Feature Engineering (Lag + Smooth)
        for col in feature_cols:
            if col == 'Cycles_Since_Last_Wash': continue # Non laggare il contatore
            
            for i in range(1, 4): 
                df_grouped[f"{col}_lag{i}"] = df_grouped.groupby('ESN')[col].shift(i)
            
            # Smoothing corto per il WW (5 voli)
            df_grouped[f"{col}_smooth"] = df_grouped.groupby('ESN')[col].transform(
                lambda x: x.rolling(window=5, min_periods=1).mean()
            )
            df_grouped[f"{col}_diff"] = df_grouped.groupby('ESN')[col].diff()

        df_grouped = df_grouped.ffill().bfill().fillna(0)
        
        # Pulizia colonne tecniche
        drop_cols = ['WW_Change', 'Wash_Session_ID', 'Cumulative_WWs']
        df_grouped = df_grouped.drop(columns=[c for c in drop_cols if c in df_grouped.columns])
        
        return df_grouped
    return pd.DataFrame()

# ==============================================================================
# 2. METRICHE & UTILS
# ==============================================================================
def get_score(y_true, y_pred, component_type):
    error = y_pred - y_true
    alpha = 0.01
    max_val = np.max(y_true) if len(y_true) > 0 else 1
    beta = 1/max_val if component_type == 'WW' else 2/max_val
    w = np.where(error >= 0, 2/(1+alpha*y_true), 1/(1+alpha*y_true))
    return np.mean(w * (error**2) * beta)

def get_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))



# ==============================================================================
# 3. MAIN
# ==============================================================================
def main():
    print("--- CARICAMENTO TRAINING (FISICA + RESIDUI) ---")
    # >>> QUI CARICHI IL TUO NUOVO FILE <<<
    try:
        # Nota: Carichiamo il file con i residui fisici se ce l'hai, 
        # oppure quello con i residui normali. La funzione prepare_... 
        # ricalcolerà comunque la fisica per sicurezza.
        df_raw = pd.read_csv('data_elaborated/train/train_with_physics_residuals.csv')
    except:
        print("File non trovato. Controllo path alternativo..."); return
    
    df_raw = harmonize_columns(df_raw)

    print("\n--- DATASET PREPARATION ---")
    df_mech = prepare_mechanical_data(df_raw)
    df_wash = prepare_wash_data(df_raw)
    
    # Configurazione Modelli
    tasks = [
        {'target': 'Cycles_to_HPT_SV', 'data': df_mech, 'type': 'HPT', 'alpha': 0.15, 'margin': 50}, 
        {'target': 'Cycles_to_HPC_SV', 'data': df_mech, 'type': 'HPC', 'alpha': 0.15, 'margin': 50},
        {'target': 'Cycles_to_WW',     'data': df_wash, 'type': 'WW',  'alpha': 0.15, 'margin': 5}  
    ]

    trained_models = {}     
    feature_sets = {}

    print("\n--- TRAINING ---")

    for task in tasks:
        target = task['target']
        df = task['data']
        ctype = task['type']
        alpha_q = task['alpha']
        margin = task['margin']
        
        if target not in df.columns: continue
        
        # Escludiamo target e colonne non-feature
        features = [c for c in df.columns if c not in ['ESN', target, 'Snapshot'] and 'Cycles_to_' not in c]
        feature_sets[ctype] = features 
        
        X = df[features]
        y = df[target]
        
        imputer = SimpleImputer(strategy='mean')
        X_clean = pd.DataFrame(imputer.fit_transform(X), columns=features)
        
        # Parametri (WW leggero, HPT/HPC robusti)
       # Parametri Ottimizzati
        if ctype == 'WW':
            # WW: Torna alla configurazione aggressiva che funzionava!
            params = {
                'n_estimators': 300,    # Aumentiamo un po'
                'max_depth': 6,         # Profondità media (prima era 4, troppo poco. 8 forse troppo)
                'learning_rate': 0.05, 
                'alpha': alpha_q, 
                'subsample': 1.0        # Usa tutti i dati (fondamentale per i lavaggi rari)
            }
        else:
            # HPT/HPC: Squadra che vince non si cambia (Score 8 e 12!)
            params = {
                'n_estimators': 300, 
                'max_depth': 2, 
                'learning_rate': 0.05, 
                'alpha': alpha_q, 
                'subsample': 0.7
            }
        print(f">>> Training {ctype} (Feats: {len(features)})...")
        model = GradientBoostingRegressor(
            loss='quantile', alpha=params['alpha'], 
            n_estimators=params['n_estimators'], 
            learning_rate=params['learning_rate'], 
            max_depth=params['max_depth'], 
            subsample=params['subsample'],
            random_state=42
        )
        model.fit(X_clean, y)
        trained_models[ctype] = {'model': model, 'margin': margin, 'target_name': target}

    # --- GENERAZIONE SUBMISSION ---
    TEST_FOLDER_PATH = 'data/val/'  
    generate_submission_file(TEST_FOLDER_PATH, trained_models, feature_sets)

   

    # --- GENERAZIONE SUBMISSION SU TEST SET (REALE) ---
   
    REAL_TEST_PATH = 'data/test/' 
    
    print(f"\n--- AVVIO GENERAZIONE SUBMISSION FINALE (Test Set) ---")
    
    # Usiamo la stessa funzione, ma cambiamo il nome del file di output
    generate_submission_file(
        REAL_TEST_PATH, 
        trained_models, 
        feature_sets, 
        output_file='submission_final_test.csv'
    )

if __name__ == "__main__":
    main()

--- CARICAMENTO TRAINING (FISICA + RESIDUI) ---

--- DATASET PREPARATION ---

--- TRAINING ---
>>> Training HPT (Feats: 13)...
>>> Training HPC (Feats: 13)...
>>> Training WW (Feats: 62)...

--- GENERAZIONE SUBMISSION FILE (48 files) ---
File salvato e pronto per l'upload: submission.csv
    file  Cycles_to_WW  Cycles_to_HPC_SV  Cycles_to_HPT_SV
0  val_0           877              3691              1131
1  val_1           834              3591              1340
2  val_2           883              3827               911
3  val_3           856              4728               327
4  val_4           861              5325               500

--- AVVIO GENERAZIONE SUBMISSION FINALE (Test Set) ---

--- GENERAZIONE SUBMISSION FILE (52 files) ---


/Users/lorenzogiannetti/Documents/GitHub/A-PHM-AMERICA-2025/MPAR/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/lorenzogiannetti/Documents/GitHub/A-PHM-AMERICA-2025/MPAR/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/lorenzogiannetti/Documents/GitHub/A-PHM-AMERICA-2025/MPAR/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


File salvato e pronto per l'upload: submission_final_test.csv
     file  Cycles_to_WW  Cycles_to_HPC_SV  Cycles_to_HPT_SV
0  test_0           876              3696              1130
1  test_1           792              5154               632
2  test_2           827              5154               632
3  test_3           895              5029              1056
4  test_4           769              5190               645


In [16]:
def main():
    print("--- CARICAMENTO TRAINING (FISICA + RESIDUI) ---")
    try:
        # Carica il file con i residui fisici se esiste, altrimenti quello normale
        path_phys = 'data_elaborated/train/train_with_physics_residuals.csv'
        if os.path.exists(path_phys):
            df_raw = pd.read_csv(path_phys)
            print(f"Usato file premium: {path_phys}")
        else:
            df_raw = pd.read_csv('data_elaborated/train/train_with_residuals.csv')
            print("Usato file standard (calcolerò la fisica al volo).")
    except:
        print("Nessun file di training trovato."); return
    
    df_raw = harmonize_columns(df_raw)

    print("\n--- DATASET PREPARATION ---")
    df_mech = prepare_mechanical_data(df_raw)
    df_wash = prepare_wash_data(df_raw)
    
    # Configurazione Modelli
    tasks = [
        {'target': 'Cycles_to_HPT_SV', 'data': df_mech, 'type': 'HPT', 'alpha': 0.15, 'margin': 50}, 
        {'target': 'Cycles_to_HPC_SV', 'data': df_mech, 'type': 'HPC', 'alpha': 0.15, 'margin': 50},
        {'target': 'Cycles_to_WW',     'data': df_wash, 'type': 'WW',  'alpha': 0.15, 'margin': 5}  
    ]

    trained_models = {}     
    feature_sets = {}

    print("\n" + "="*60)
    print(" AVVIO TRAINING FINALE E VERIFICA PERFORMANCE")
    print("="*60)

    for task in tasks:
        target = task['target']
        df = task['data']
        ctype = task['type']
        alpha_q = task['alpha']
        margin = task['margin']
        
        if target not in df.columns: continue
        
        # Selezione Feature
        features = [c for c in df.columns if c not in ['ESN', target, 'Snapshot'] and 'Cycles_to_' not in c]
        feature_sets[ctype] = features 
        
        X = df[features]
        y = df[target]
        
        # Imputer
        imputer = SimpleImputer(strategy='mean')
        X_clean = pd.DataFrame(imputer.fit_transform(X), columns=features)
        
        # Parametri Ottimizzati
       # Parametri Ottimizzati
        if ctype == 'WW':
            # WW: Torna alla configurazione aggressiva che funzionava!
            params = {
                'n_estimators': 300,    # Aumentiamo un po'
                'max_depth': 6,         # Profondità media (prima era 4, troppo poco. 8 forse troppo)
                'learning_rate': 0.05, 
                'alpha': alpha_q, 
                'subsample': 1.0        # Usa tutti i dati (fondamentale per i lavaggi rari)
            }
        else:
            # HPT/HPC: Squadra che vince non si cambia (Score 8 e 12!)
            params = {
                'n_estimators': 300, 
                'max_depth': 2, 
                'learning_rate': 0.05, 
                'alpha': alpha_q, 
                'subsample': 0.7
            }
        print(f"\n>>> Modello {ctype} (Feats: {len(features)})")
        
        # 1. Addestramento
        model = GradientBoostingRegressor(
            loss='quantile', alpha=params['alpha'], 
            n_estimators=params['n_estimators'], 
            learning_rate=params['learning_rate'], 
            max_depth=params['max_depth'], 
            subsample=params['subsample'],
            random_state=42
        )
        model.fit(X_clean, y)
        
        # 2. VERIFICA PERFORMANCE (Training Score) <--- AGGIUNTO QUESTO
        pred_train = model.predict(X_clean)
        # Applichiamo il margine di sicurezza anche qui per vedere il punteggio reale
        pred_train_safe = np.maximum(pred_train - margin, 0)
        
        train_rmse = get_rmse(y, pred_train_safe)
        train_score = get_score(y, pred_train_safe, ctype)
        
        print(f"    [TRAIN RESULT] RMSE: {train_rmse:.2f} | Score: {train_score:.4f}")
        
        trained_models[ctype] = {'model': model, 'margin': margin, 'target_name': target}

    # --- GENERAZIONE SUBMISSION ---
    print("\n" + "="*60)
    
    # 1. Validation Set
    TEST_FOLDER_PATH = 'data/val/'
    if os.path.exists(TEST_FOLDER_PATH):
        generate_submission_file(TEST_FOLDER_PATH, trained_models, feature_sets, output_file='submission_residuals_validation.csv')
    
    # 2. Real Test Set
    REAL_TEST_PATH = 'data/test/' 
    if os.path.exists(REAL_TEST_PATH):
         generate_submission_file(REAL_TEST_PATH, trained_models, feature_sets, output_file='submission_test_residuals.csv')

if __name__ == "__main__":
    main()

--- CARICAMENTO TRAINING (FISICA + RESIDUI) ---
Usato file premium: data_elaborated/train/train_with_physics_residuals.csv

--- DATASET PREPARATION ---

 AVVIO TRAINING FINALE E VERIFICA PERFORMANCE

>>> Modello HPT (Feats: 13)
    [TRAIN RESULT] RMSE: 685.02 | Score: 8.7204

>>> Modello HPC (Feats: 13)
    [TRAIN RESULT] RMSE: 1687.15 | Score: 12.1267

>>> Modello WW (Feats: 62)
    [TRAIN RESULT] RMSE: 83.84 | Score: 0.9943


--- GENERAZIONE SUBMISSION FILE (48 files) ---
File salvato e pronto per l'upload: submission_residuals_validation.csv
    file  Cycles_to_WW  Cycles_to_HPC_SV  Cycles_to_HPT_SV
0  val_0           877              3691              1131
1  val_1           834              3591              1340
2  val_2           883              3827               911
3  val_3           856              4728               327
4  val_4           861              5325               500

--- GENERAZIONE SUBMISSION FILE (52 files) ---


/Users/lorenzogiannetti/Documents/GitHub/A-PHM-AMERICA-2025/MPAR/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/lorenzogiannetti/Documents/GitHub/A-PHM-AMERICA-2025/MPAR/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/lorenzogiannetti/Documents/GitHub/A-PHM-AMERICA-2025/MPAR/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


File salvato e pronto per l'upload: submission_test_residuals.csv
     file  Cycles_to_WW  Cycles_to_HPC_SV  Cycles_to_HPT_SV
0  test_0           876              3696              1130
1  test_1           792              5154               632
2  test_2           827              5154               632
3  test_3           895              5029              1056
4  test_4           769              5190               645
